# Unified RAD-DINO pathology retrieval pipeline

Enable a GPU accelerator and attach these Kaggle Datasets: `image-rag-project`, `iu-xray-reprocessed`, and `rad-dino-raw-embeddings`. Create a Kaggle Secret named `HF_TOKEN` before running.

In [ ]:
from pathlib import Path
import os
import shutil

# Change only these paths if your Kaggle Dataset slugs differ.
PROJECT_SOURCE_INPUT = Path('/kaggle/input/image-rag-project')
PROCESSED_INPUT = Path('/kaggle/input/iu-xray-reprocessed/reprocessed')
RAW_EMBEDDINGS_INPUT = Path('/kaggle/input/rad-dino-raw-embeddings/embeddings')
WORKING_DIR = Path('/kaggle/working/rad-dino-pathology')
REPO_DIR = WORKING_DIR / 'project'

assert (PROJECT_SOURCE_INPUT / 'scripts' / 'train_pathology_projection.py').is_file()
assert (PROCESSED_INPUT / 'metadata.jsonl').is_file()
assert (PROCESSED_INPUT / 'images').is_dir()
for filename in ('embeddings.npy', 'ids.txt', 'metadata.jsonl', 'embedding_manifest.json'):
    assert (RAW_EMBEDDINGS_INPUT / filename).is_file(), filename

shutil.rmtree(WORKING_DIR, ignore_errors=True)
shutil.copytree(PROJECT_SOURCE_INPUT, REPO_DIR)
os.chdir(REPO_DIR)
print(REPO_DIR)

In [ ]:
import subprocess
import sys
import torch

assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator before running this notebook.'
print(torch.cuda.get_device_name(0))

# Keep Kaggle's CUDA-enabled torch wheels; install the remaining runtime dependencies.
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.38', 'python-dotenv>=1.0', 'PyYAML>=6.0',
    'tqdm>=4.66', 'faiss-cpu>=1.7'
])

In [ ]:
from kaggle_secrets import UserSecretsClient

os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
assert os.environ['HF_TOKEN'], 'Kaggle Secret HF_TOKEN is required.'
print('HF_TOKEN loaded from Kaggle Secrets.')

In [ ]:
CHECKPOINT_PATH = WORKING_DIR / 'pathology_projection_rad_dino.pt'

train_command = [
    sys.executable, 'scripts/train_pathology_projection.py',
    '--embeddings_path', str(RAW_EMBEDDINGS_INPUT / 'embeddings.npy'),
    '--metadata_path', str(RAW_EMBEDDINGS_INPUT / 'metadata.jsonl'),
    '--processed_dir', str(PROCESSED_INPUT),
    '--output_checkpoint', str(CHECKPOINT_PATH),
    '--epochs', '30', '--batch_size', '16',
    '--lr', '1e-4', '--encoder_lr', '1e-5', '--device', 'cuda',
]
subprocess.check_call(train_command, cwd=REPO_DIR)

In [ ]:
EMBEDDING_OUTPUT = WORKING_DIR / 'embeddings_pathology'

embed_command = [
    sys.executable, 'scripts/compute_image_embeddings.py',
    '--processed_dir', str(PROCESSED_INPUT),
    '--out_dir', str(EMBEDDING_OUTPUT),
    '--projection_checkpoint', str(CHECKPOINT_PATH),
    '--vector_db', 'faiss', '--clean', '--batch_size', '8', '--device', 'cuda',
]
subprocess.check_call(embed_command, cwd=REPO_DIR)

In [ ]:
# Preserve MeSH and Problems even when the processed-image Dataset was created before metadata restoration.
sync_command = [
    sys.executable, 'scripts/sync_retrieval_metadata.py',
    '--source_metadata', str(RAW_EMBEDDINGS_INPUT / 'metadata.jsonl'),
    '--target_metadata', str(EMBEDDING_OUTPUT / 'metadata.jsonl'),
]
subprocess.check_call(sync_command, cwd=REPO_DIR)

In [ ]:
EVALUATION_OUTPUT = WORKING_DIR / 'evaluation'
evaluate_command = [
    sys.executable, 'scripts/evaluate_pathology_retrieval.py',
    '--gallery_dir', str(EMBEDDING_OUTPUT),
    '--output_dir', str(EVALUATION_OUTPUT),
    '--k', '5', '10', '--relevance_threshold', '0.5',
]
subprocess.check_call(evaluate_command, cwd=REPO_DIR)

In [ ]:
BUNDLE_DIR = WORKING_DIR / 'rad_dino_pathology_output'
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(CHECKPOINT_PATH, BUNDLE_DIR / CHECKPOINT_PATH.name)
shutil.copytree(EMBEDDING_OUTPUT, BUNDLE_DIR / 'embeddings_pathology', dirs_exist_ok=True)
shutil.copytree(EVALUATION_OUTPUT, BUNDLE_DIR / 'evaluation', dirs_exist_ok=True)
archive_path = shutil.make_archive(str(WORKING_DIR / 'rad_dino_pathology_output'), 'zip', root_dir=BUNDLE_DIR)
print(f'Download this file: {archive_path}')